# 02 — Pregătire Dataset Clasificare

Acest notebook acoperă **Stage 2 (clasificare materiale)** — pregătire date:

0. Generare crops + auto-etichetare cu detectorul A22 + clasificatorul B2
1. Split dataset clasificare (crops din parks)
2. Statistici dataset clasificare
3. Merge dataset-uri clasificare (TrashNet + parks crops → mixed_cls)

**Pre-condiție**: Ai rulat **01_data_preparation.ipynb** și ai imagini în `datasets/parks_detect/images/all`.

Clasele: `glass | metal | other | paper | plastic`


---
## 0. Generare Crops + Auto-Etichetare cu A22 + B2

Pasul acesta este **pre-condiția** pentru Secțiunile 1-3 și se rulează o singură dată.

**Pipeline:**
1. Rulează **detectorul A22** pe toate imaginile de parc (`parks_detect/images/`) → labels YOLO de predicție  
2. Exportă **crop-urile** (obiectele detectate) din imagini  
3. **Auto-clasifică** fiecare crop cu **clasificatorul B2** → sortează în subfolderele de clasă

> Pseudo-labels generate de B2 (antrenat pe TrashNet) vor fi zgomotoase (~80-90% precizie).  
> Poți valida vizual rezultatele după rulare.


In [1]:
import sys
import shutil
from pathlib import Path
import csv

from ultralytics import YOLO

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLASSES    = ["glass", "metal", "other", "paper", "plastic"]

# ── CONFIG auto-labeling ──────────────────────────────────────────────────────
DETECTOR_PT   = REPO_ROOT / "runs" / "detect"   / "parks-trash-A22" / "weights" / "best.pt"
CLASSIFIER_PT = REPO_ROOT / "runs" / "classify" / "parks-cls-B2"    / "weights" / "best.pt"

# Imaginile sursă pentru crop-uri
IMAGES_DIR    = REPO_ROOT / "datasets" / "parks_detect" / "images" / "all"

# Predicții detector → labels temp
PRED_LABELS_DIR = REPO_ROOT / "datasets" / "parks_cls_unsorted" / "pred_labels"

# Crops flat (nesortate) + manifest
CROPS_FLAT_DIR  = REPO_ROOT / "datasets" / "parks_cls_unsorted" / "all_flat"
MANIFEST_PATH   = REPO_ROOT / "datasets" / "parks_cls_unsorted" / "crops_manifest.csv"

# Crops sortate pe clase (input pentru split)
CROPS_SORTED_DIR = REPO_ROOT / "datasets" / "parks_cls_unsorted" / "all"

# Prag detector
DET_CONF   = 0.15   # mai mic → mai multe crops (pot fi FP, se filtrează de B2)
DET_IMGSZ  = 416
CLS_IMGSZ  = 224
DEVICE     = "0"
MARGIN     = 0.05   # expansiune crop (5%)

print(f"Detector : {DETECTOR_PT}")
print(f"Clasificator: {CLASSIFIER_PT}")
print(f"Imagini  : {IMAGES_DIR}")
assert DETECTOR_PT.exists(),   f"Lipsă detector: {DETECTOR_PT}"
assert CLASSIFIER_PT.exists(), f"Lipsă clasificator: {CLASSIFIER_PT}"


Detector : D:\TrashDetectionSystem\runs\detect\parks-trash-A22\weights\best.pt
Clasificator: D:\TrashDetectionSystem\runs\classify\parks-cls-B2\weights\best.pt
Imagini  : D:\TrashDetectionSystem\datasets\parks_detect\images\all


In [2]:
import cv2

def expand_box(x1, y1, x2, y2, margin, W, H):
    pw, ph = (x2 - x1) * margin, (y2 - y1) * margin
    return (max(int(x1 - pw), 0), max(int(y1 - ph), 0),
            min(int(x2 + pw), W),  min(int(y2 + ph), H))

# ── Pasul 1: Rulare detector A22 pe toate imaginile ──────────────────────────
PRED_LABELS_DIR.mkdir(parents=True, exist_ok=True)
CROPS_FLAT_DIR.mkdir(parents=True, exist_ok=True)

detector   = YOLO(str(DETECTOR_PT))
classifier = YOLO(str(CLASSIFIER_PT))
cls_names  = {int(k): str(v) for k, v in (classifier.names.items()
              if isinstance(classifier.names, dict) else enumerate(classifier.names))}
print(f"Clase clasificator: {cls_names}")

images = sorted(p for p in IMAGES_DIR.rglob("*")
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
print(f"Imagini de procesat: {len(images)}")

manifest_rows = []
crop_counter = 0

for img_path in images:
    frame = cv2.imread(str(img_path))
    if frame is None:
        continue
    H, W = frame.shape[:2]

    det_results = detector.predict(
        str(img_path), conf=DET_CONF, imgsz=DET_IMGSZ,
        device=DEVICE, verbose=False
    )[0]

    boxes = det_results.boxes
    if boxes is None or len(boxes) == 0:
        continue

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = (int(v) for v in box.xyxy[0].tolist())
        det_score = float(box.conf[0])
        lx, ty, rx, by = expand_box(x1, y1, x2, y2, MARGIN, W, H)
        crop = frame[ty:by, lx:rx]
        if crop.size == 0:
            continue

        crop_name = f"{img_path.stem}_crop_{i:03d}.jpg"
        crop_path = CROPS_FLAT_DIR / crop_name
        cv2.imwrite(str(crop_path), crop)

        # ── Pasul 2: Clasificare cu B2 ─────────────────────────────────────
        cls_result = classifier.predict(str(crop_path), imgsz=CLS_IMGSZ,
                                        device=DEVICE, verbose=False)[0]
        probs = cls_result.probs
        cls_idx  = int(probs.top1)
        cls_conf = float(probs.top1conf.item() if hasattr(probs.top1conf, "item") else probs.top1conf)
        cls_name = cls_names.get(cls_idx, str(cls_idx))

        # ── Pasul 3: Sortare în subdirector de clasă ───────────────────────
        sorted_dir = CROPS_SORTED_DIR / cls_name
        sorted_dir.mkdir(parents=True, exist_ok=True)
        final_path = sorted_dir / crop_name
        shutil.copy2(crop_path, final_path)

        manifest_rows.append({
            "crop_path":   str(final_path),
            "source_image": str(img_path),
            "box_index":   i,
            "det_score":   round(det_score, 4),
            "cls_name":    cls_name,
            "cls_score":   round(cls_conf, 4),
            "x1": lx, "y1": ty, "x2": rx, "y2": by,
        })
        crop_counter += 1

print(f"\nCrops generate: {crop_counter}")

# Scrie manifest CSV actualizat
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
fieldnames = ["crop_path", "source_image", "box_index",
              "det_score", "cls_name", "cls_score", "x1", "y1", "x2", "y2"]
with MANIFEST_PATH.open("w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(manifest_rows)
print(f"Manifest salvat: {MANIFEST_PATH}  ({len(manifest_rows)} înregistrări)")


Clase clasificator: {0: 'glass', 1: 'metal', 2: 'other', 3: 'paper', 4: 'plastic'}
Imagini de procesat: 48

Crops generate: 19
Manifest salvat: D:\TrashDetectionSystem\datasets\parks_cls_unsorted\crops_manifest.csv  (19 înregistrări)


In [3]:
from collections import Counter

# Sumar distribuție clase
class_counts = Counter(r["cls_name"] for r in manifest_rows)
print("Distribuție clase (pseudo-labels):")
for cls in CLASSES:
    print(f"  {cls:<10} {class_counts.get(cls, 0):>5}")

# Distribuție scoruri clasificare
scores = [r["cls_score"] for r in manifest_rows]
if scores:
    import statistics
    print(f"\nScor clasificare  min={min(scores):.3f}  med={statistics.median(scores):.3f}  max={max(scores):.3f}")
    low_conf = sum(1 for s in scores if s < 0.5)
    print(f"Crops cu scor < 0.5 (incerte): {low_conf}/{len(scores)}")


Distribuție clase (pseudo-labels):
  glass          2
  metal          5
  other          0
  paper         10
  plastic        2

Scor clasificare  min=0.390  med=0.922  max=0.998
Crops cu scor < 0.5 (incerte): 4/19


In [4]:
import random
import shutil
import sys
import csv
from collections import defaultdict
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLASSES    = ["glass", "metal", "other", "paper", "plastic"]

print(f"Rădăcina proiectului: {REPO_ROOT}")

Rădăcina proiectului: D:\TrashDetectionSystem


In [5]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────
# Sursă: crops sortate pe clase de la Secțiunea 0
# Dacă ai rulat Secțiunea 0, CROPS_SORTED_DIR și MANIFEST_PATH sunt deja definite.
# Altfel, setează manual:
# CROPS_SORTED_DIR = REPO_ROOT / "datasets" / "parks_cls_unsorted" / "all"
# MANIFEST_PATH    = REPO_ROOT / "datasets" / "parks_cls_unsorted" / "crops_manifest.csv"

CLS_SOURCE_ROOT  = CROPS_SORTED_DIR
CLS_OUT_ROOT     = REPO_ROOT / "datasets" / "parks_cls"
CLS_VAL_RATIO    = 0.15
CLS_TEST_RATIO   = 0.15
CLS_SEED         = 42
CLS_GROUP_BY     = "source-image"   # "source-image" sau "crop"
CLS_COPY         = True
CLS_CLEAR        = True

# Merge clasificare
TRASHNET_DIR     = REPO_ROOT / "datasets" / "trashnet_cls"
MIXED_OUT_DIR    = REPO_ROOT / "datasets" / "mixed_cls"

print("Config OK")


Config OK


---
## 1. Split Dataset Clasificare

Împarte crops-urile adnotate din `parks_cls_unsorted` în train/val/test.  
Gruparea pe `source-image` previne leakage-ul — crops din aceeași imagine sursă nu vor fi în split-uri diferite.

> **Notă**: Structura așteptată a sursei este `parks_cls_unsorted/all/<clasa>/imagine.jpg`  
> Dacă ai crops nesortate și le-ai sortat manual pe clase, rulează această celulă.

In [6]:
def iter_images(directory: Path):
    if not directory.exists(): return []
    return sorted(p for p in directory.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def load_manifest(manifest_path: Path):
    if not manifest_path.exists(): return {}
    mapping = {}
    with manifest_path.open("r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            mapping[Path(row["crop_path"]).name] = Path(row["source_image"]).stem
    return mapping


def assign_groups(group_keys, val_ratio, test_ratio, seed):
    shuffled = list(group_keys)
    random.Random(seed).shuffle(shuffled)
    total = len(shuffled)
    tc = int(round(total * test_ratio)); vc = int(round(total * val_ratio))
    if total >= 3:
        if test_ratio > 0 and tc == 0: tc = 1
        if val_ratio  > 0 and vc == 0: vc = 1
        ov = tc + vc - total + 1
        while ov > 0 and tc > 0: tc -= 1; ov -= 1
        while ov > 0 and vc > 0: vc -= 1; ov -= 1
    sm = {}
    for i, k in enumerate(shuffled):
        sm[k] = "test" if i < tc else ("val" if i < tc + vc else "train")
    return sm


def split_classification_dataset(
    source_root, out_root, manifest_path, classes,
    val_ratio, test_ratio, seed, group_by, copy_files, clear
):
    source_root = Path(source_root); out_root = Path(out_root)
    if not source_root.exists():
        print(f"[EROARE] Sursă inexistentă: {source_root}"); return

    manifest_map = load_manifest(Path(manifest_path)) if group_by == "source-image" else {}

    grouped_items = defaultdict(list)
    for cls in classes:
        cls_dir = source_root / cls
        if not cls_dir.exists(): continue
        for img in iter_images(cls_dir):
            key = manifest_map.get(img.name, img.stem) if group_by == "source-image" else img.stem
            grouped_items[(cls, key)].append(img)

    if not grouped_items:
        print(f"[EROARE] Nicio imagine în {source_root}"); return

    if clear:
        for split in ("train", "val", "test"):
            d = out_root / split
            if d.exists(): shutil.rmtree(d)

    for split in ("train", "val", "test"):
        for cls in classes:
            (out_root / split / cls).mkdir(parents=True, exist_ok=True)

    grouped_by_class = defaultdict(list)
    for (cls, key) in grouped_items:
        grouped_by_class[cls].append(key)

    stats = {sp: defaultdict(int) for sp in ("train","val","test")}
    for cls in classes:
        keys = grouped_by_class.get(cls, [])
        if not keys: continue
        sm = assign_groups(keys, val_ratio, test_ratio, seed)
        for key in keys:
            sp = sm[key]
            for src in grouped_items[(cls, key)]:
                dst = out_root / sp / cls / src.name
                dst.parent.mkdir(parents=True, exist_ok=True)
                if copy_files: shutil.copy2(src, dst)
                else: shutil.move(str(src), str(dst))
                stats[sp][cls] += 1

    print(f"{'Clasa':<12} {'Train':>7} {'Val':>7} {'Test':>7} {'Total':>8}")
    print("-" * 45)
    for cls in classes:
        tr = stats['train'][cls]; va = stats['val'][cls]; te = stats['test'][cls]
        print(f"{cls:<12} {tr:>7} {va:>7} {te:>7} {tr+va+te:>8}")
    print("\nSplit clasificare generat!")


split_classification_dataset(
    CLS_SOURCE_ROOT, CLS_OUT_ROOT, MANIFEST_PATH, CLASSES,
    CLS_VAL_RATIO, CLS_TEST_RATIO, CLS_SEED, CLS_GROUP_BY, CLS_COPY, CLS_CLEAR
)

Clasa          Train     Val    Test    Total
---------------------------------------------
glass              8       2       2       12
metal             64      14      14       92
other              1       1       1        3
paper             96      20      20      136
plastic           29       6       6       41

Split clasificare generat!


---
## 2. Statistici Dataset Clasificare
Afișează distribuția imaginilor pe clase și split-uri.

In [7]:
def report_classification_stats(data_root: Path, classes):
    if not data_root.exists():
        print(f"[EROARE] Nu există: {data_root}"); return

    def count_imgs(d): return sum(1 for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS) if d.exists() else 0

    print(f"Dataset: {data_root.name}")
    print(f"{'Clasa':<12} {'Train':>7} {'Val':>7} {'Test':>7} {'Total':>8}")
    print("-" * 45)

    grand = {"train": 0, "val": 0, "test": 0}
    for cls in classes:
        tr = count_imgs(data_root / "train" / cls)
        va = count_imgs(data_root / "val"   / cls)
        te = count_imgs(data_root / "test"  / cls)
        print(f"{cls:<12} {tr:>7} {va:>7} {te:>7} {tr+va+te:>8}")
        grand["train"] += tr; grand["val"] += va; grand["test"] += te

    print("-" * 45)
    print(f"{'TOTAL':<12} {grand['train']:>7} {grand['val']:>7} {grand['test']:>7} {sum(grand.values()):>8}")


print("=== parks_cls ===")
report_classification_stats(REPO_ROOT / "datasets" / "parks_cls", CLASSES)

=== parks_cls ===
Dataset: parks_cls
Clasa          Train     Val    Test    Total
---------------------------------------------
glass              8       2       2       12
metal             64      14      14       92
other              1       1       1        3
paper             96      20      20      136
plastic           29       6       6       41
---------------------------------------------
TOTAL            198      43      43      284


In [8]:
print("=== trashnet_cls ===")
report_classification_stats(TRASHNET_DIR, CLASSES)
print()
print("=== mixed_cls ===")
report_classification_stats(MIXED_OUT_DIR, CLASSES)

=== trashnet_cls ===
Dataset: trashnet_cls
Clasa          Train     Val    Test    Total
---------------------------------------------
glass            400      50      51      501
metal            328      41      41      410
other            109      13      15      137
paper            797      99     101      997
plastic          385      48      49      482
---------------------------------------------
TOTAL           2019     251     257     2527

=== mixed_cls ===
Dataset: mixed_cls
Clasa          Train     Val    Test    Total
---------------------------------------------
glass            407      53      51      511
metal            390      53      59      502
other            110      14      16      140
paper            898     117     118     1133
plastic          416      54      53      523
---------------------------------------------
TOTAL           2221     291     297     2809


---
## 3. Merge Dataset-uri Clasificare
Combină **TrashNet** + **parks_cls** în `mixed_cls` (folosit pentru Experimentul B3 din experiment plan).  

**Pre-condiție**: Ai rulat `scripts/download_trashnet.py` pentru a descărca TrashNet.

In [9]:
def merge_classification_datasets(sources: list, out_dir: Path, classes):
    sources = [Path(s) for s in sources]
    for src in sources:
        if not src.exists():
            print(f"[WARN] Sursă inexistentă: {src}")

    if out_dir.exists():
        print(f"Șterg output existent: {out_dir}")
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)

    totals = defaultdict(lambda: defaultdict(int))

    for src in sources:
        if not src.exists(): continue
        for split in ("train", "val", "test"):
            for cls in classes:
                src_cls = src / split / cls
                if not src_cls.exists(): continue
                dst_cls = out_dir / split / cls
                dst_cls.mkdir(parents=True, exist_ok=True)
                imgs = list(src_cls.glob("*.jpg")) + list(src_cls.glob("*.png")) + list(src_cls.glob("*.jpeg"))
                for img in imgs:
                    dst = dst_cls / f"{src.name}_{img.name}"
                    shutil.copy2(img, dst)
                    totals[cls][split] += 1
        print(f"Procesat: {src.name}")

    print(f"\nDataset merged: {out_dir}")
    print(f"{'Clasa':<12} {'Train':>7} {'Val':>7} {'Test':>7} {'Total':>8}")
    print("-" * 45)
    for cls in classes:
        tr = totals[cls]["train"]; va = totals[cls]["val"]; te = totals[cls]["test"]
        print(f"{cls:<12} {tr:>7} {va:>7} {te:>7} {tr+va+te:>8}")


merge_classification_datasets(
    [TRASHNET_DIR, REPO_ROOT / "datasets" / "parks_cls"],
    MIXED_OUT_DIR,
    CLASSES
)

Șterg output existent: D:\TrashDetectionSystem\datasets\mixed_cls
Procesat: trashnet_cls
Procesat: parks_cls

Dataset merged: D:\TrashDetectionSystem\datasets\mixed_cls
Clasa          Train     Val    Test    Total
---------------------------------------------
glass            408      52      53      513
metal            392      55      55      502
other            110      14      16      140
paper            893     119     121     1133
plastic          414      54      55      523
